# K=3 90-to-30 supervised experiment — five strides
Enable a GPU and attach this package, the stride-1 dataset, and the strides 5/10/15/30 dataset.

In [1]:
from pathlib import Path

root = Path("/kaggle/input/datasets")

print("ATTACHED DATASET DIRECTORIES\n")

for path in sorted(root.glob("*/*")):
    if path.is_dir():
        print(path)

ATTACHED DATASET DIRECTORIES

/kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-forecasting-dataset-v1
/kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-stride1-forecasting-dataset-v1
/kaggle/input/datasets/mozimandias/k3-90to30-supervised-learning-five-strides-v1-1


In [2]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before Layer-4 training.'

CUDA available: True
GPU: Tesla T4


In [3]:
from pathlib import Path
import shutil

matches = list(Path('/kaggle/input').rglob('k3_90to30_supervised_learning_v1_1/configs/training_config.json'))
assert len(matches) == 1, f'Expected one package, found: {matches}'
source = matches[0].parents[1]
package = Path('/kaggle/working/k3_90to30_supervised_learning_v1_1')
if not package.exists(): shutil.copytree(source, package)
config_path = package / 'configs/training_config.json'
print('Working package:', package)

Working package: /kaggle/working/k3_90to30_supervised_learning_v1_1


## Dependency check
Kaggle supplies CUDA-enabled PyTorch and torchvision. Pin only the scikit-learn version used for metrics.

In [4]:
!pip install -q scikit-learn==1.6.1

import subprocess, sys
subprocess.run([sys.executable, '-c', "import sklearn; assert sklearn.__version__ == '1.6.1'; print('Subprocess scikit-learn:', sklearn.__version__)"], check=True)

Subprocess scikit-learn: 1.6.1


CompletedProcess(args=['/usr/bin/python3', '-c', "import sklearn; assert sklearn.__version__ == '1.6.1'; print('Subprocess scikit-learn:', sklearn.__version__)"], returncode=0)

## Configuration
The package is locked to a primary comparison on strides 5/10/15/30 and a separate stride-1 sensitivity extension.

In [5]:
import json
config = json.loads(config_path.read_text())
print(json.dumps(config, indent=2))

{
  "dataset_roots": {
    "1": null,
    "5": null,
    "10": null,
    "15": null,
    "30": null
  },
  "search_root": "/kaggle/input",
  "output_dir": "/kaggle/working/k3_90to30_supervised_results_v1_1",
  "strides": [
    1,
    5,
    10,
    15,
    30
  ],
  "primary_strides": [
    5,
    10,
    15,
    30
  ],
  "sensitivity_strides": [
    1
  ],
  "models": [
    "linear",
    "mlp",
    "layer4"
  ],
  "random_seed": 42,
  "num_workers": 2,
  "amp": true,
  "use_inverse_soft_class_mass_weights": true,
  "use_certainty_weights": true,
  "selection_metric": "macro_f1",
  "embedding_batch_size": 128,
  "embedding_models": {
    "linear": {
      "epochs": 50,
      "patience": 8,
      "batch_size": 512,
      "learning_rate": 0.001,
      "weight_decay": 0.0001
    },
    "mlp": {
      "epochs": 40,
      "patience": 8,
      "batch_size": 256,
      "learning_rate": 0.0003,
      "weight_decay": 0.0001,
      "hidden_dimensions": [
        256,
        128
      ],
      

In [6]:
!python {package/'k3_90to30_supervised/preflight.py'} --config {config_path}

CUDA available: True
GPU: Tesla T4
stride 1 dataset: /kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-stride1-forecasting-dataset-v1/BTCUSDT_k3_90to30_stride1_dataset_v1_0
stride 5 dataset: /kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-forecasting-dataset-v1/BTCUSDT_k3_90to30_dataset_v1_0
stride 10 dataset: /kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-forecasting-dataset-v1/BTCUSDT_k3_90to30_dataset_v1_0
stride 15 dataset: /kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-forecasting-dataset-v1/BTCUSDT_k3_90to30_dataset_v1_0
stride 30 dataset: /kaggle/input/datasets/mozimandias/btcusdt-k3-90to30-forecasting-dataset-v1/BTCUSDT_k3_90to30_dataset_v1_0
stride 1: train=41,199 validation=4,715 test_manifest_sha256=0cf4f3a2f551952841d99e10be097a76ca333ccddb63ee5e8cc170af821a9432
  train hard counts: {0: 7051, 1: 27141, 2: 7007}
  train soft mass: {'0': 10391.661, '1': 21033.867, '2': 9773.473}
  training class weights: [1.32154, 0.652899, 1.40513]
stride 5: train=8,233 v

## Validation-only training and selection
This cell does not load test labels. It trains all requested stride/model combinations and writes the pre-test freeze record.

In [7]:
!python {package/'k3_90to30_supervised/run_validation_experiment.py'} --config {config_path}

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 180MB/s]
Frozen ResNet-18 embeddings: 100%|████████████| 359/359 [03:18<00:00,  1.81it/s]
/kaggle/working/k3_90to30_supervised_learning_v1_1/k3_90to30_supervised/run_validation_experiment.py:168: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
stride 1 linear epoch 1:   0%|                           | 0/81 [00:00<?, ?it/s]/kaggle/working/k3_90to30_supervised_learning_v1_1/k3_90to30_supervised/run_validation_experiment.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
stride 1 linear epoch 1: 100%|█████████

In [8]:
from pathlib import Path
import hashlib
import json

output = Path(config["output_dir"])
freeze_path = output / "PRE_TEST_FREEZE.json"
completion_lock = output / "TEST_EVALUATION_COMPLETE.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

freeze = json.loads(freeze_path.read_text())

expected = {
    (stride, model)
    for stride in [1, 5, 10, 15, 30]
    for model in ["linear", "mlp", "layer4"]
}

observed = {
    (int(item["stride"]), item["model_type"])
    for item in freeze["checkpoints"]
}

checkpoint_checks = []

for item in freeze["checkpoints"]:
    path = Path(item["path"])
    checkpoint_checks.append(
        path.is_file() and sha256_file(path) == item["sha256"]
    )

test_manifest_checks = []

for stride, item in freeze["test_manifests"].items():
    path = Path(item["path"])
    test_manifest_checks.append(
        path.is_file() and sha256_file(path) == item["sha256"]
    )

print("Freeze exists:", freeze_path.is_file())
print("Freeze status:", freeze["status"])
print("Completion lock exists:", completion_lock.exists())
print("Configuration hash matches:",
      sha256_file(config_path) == freeze["configuration_sha256"])
print("Checkpoint count:", len(freeze["checkpoints"]))
print("All 15 configurations present:", observed == expected)
print("All checkpoint hashes match:", all(checkpoint_checks))
print("Test manifest count:", len(freeze["test_manifests"]))
print("All test manifest hashes match:", all(test_manifest_checks))
print("Test labels used during selection:",
      freeze["test_labels_loaded_during_training_or_selection"])
print("Test already evaluated:", freeze["test_evaluated"])
print("Primary winner:", freeze["selected_primary_winner"])
print("Stride-1 model:", freeze["selected_stride1_model"])
print("Freeze SHA-256:", sha256_file(freeze_path))

Freeze exists: True
Freeze status: pre_test_frozen
Completion lock exists: False
Configuration hash matches: True
Checkpoint count: 15
All 15 configurations present: True
All checkpoint hashes match: True
Test manifest count: 5
All test manifest hashes match: True
Test labels used during selection: False
Test already evaluated: False
Primary winner: {'stride': 15, 'model_type': 'layer4', 'validation_metrics': {'accuracy': 0.5095541401273885, 'balanced_accuracy': 0.40830280830280835, 'macro_f1': 0.3979159587722279, 'weighted_f1': 0.5331472066936334, 'soft_cross_entropy': 1.0930430889129639, 'soft_brier_score': 0.048038095235824585, 'confusion_matrix': [[17, 26, 9], [49, 128, 33], [17, 20, 15]], 'per_class': {'0': {'precision': 0.20481927710843373, 'recall': 0.3269230769230769, 'f1': 0.2518518518518518, 'support': 52}, '1': {'precision': 0.735632183908046, 'recall': 0.6095238095238096, 'f1': 0.6666666666666666, 'support': 210}, '2': {'precision': 0.2631578947368421, 'recall': 0.288461538

## Inspect the freeze record
Confirm that all expected checkpoints are listed before continuing.

In [ ]:
import json
freeze_path = Path('/kaggle/working/k3_90to30_supervised_results_v1_1/PRE_TEST_FREEZE.json')
freeze = json.loads(freeze_path.read_text())
print(json.dumps(freeze, indent=2))

## One-time comparative test evaluation
Run exactly once. The completion lock prevents an accidental rerun. These results are comparative post-test evidence because the period was already examined in the 30-to-30 phase.

In [9]:
!python {package/'k3_90to30_supervised/evaluate_frozen_suite.py'} --config {config_path}

Frozen test embeddings: 100%|█████████████████████| 3/3 [00:01<00:00,  1.73it/s]

COMPARATIVE TEST EVALUATION COMPLETE
Selected primary winner test metrics: {'accuracy': 0.4383116883116883, 'balanced_accuracy': 0.34795321637426896, 'macro_f1': 0.34617366875778893, 'weighted_f1': 0.4458157118634909, 'soft_cross_entropy': 1.1023796796798706, 'soft_brier_score': 0.04998713359236717, 'confusion_matrix': [[29, 72, 15], [90, 213, 77], [23, 69, 28]], 'per_class': {'0': {'precision': 0.20422535211267606, 'recall': 0.25, 'f1': 0.2248062015503876, 'support': 116}, '1': {'precision': 0.6016949152542372, 'recall': 0.5605263157894737, 'f1': 0.5803814713896458, 'support': 380}, '2': {'precision': 0.23333333333333334, 'recall': 0.23333333333333334, 'f1': 0.23333333333333334, 'support': 120}}}
Selected stride-1 model test metrics: {'accuracy': 0.482523536413808, 'balanced_accuracy': 0.3470302795533747, 'macro_f1': 0.3429651972907029, 'weighted_f1': 0.4707199040857028, 'soft_cross_entropy': 1.107639908

In [10]:
from pathlib import Path
import hashlib
import json

output = Path(config["output_dir"])

freeze_path = output / "PRE_TEST_FREEZE.json"
suite_path = output / "comparative_test_suite.json"
lock_path = output / "TEST_EVALUATION_COMPLETE.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

freeze = json.loads(freeze_path.read_text())
suite = json.loads(suite_path.read_text())
lock = json.loads(lock_path.read_text())

expected = {
    (stride, model)
    for stride in [1, 5, 10, 15, 30]
    for model in ["linear", "mlp", "layer4"]
}

observed = {
    (int(item["stride"]), item["model_type"])
    for item in suite["all_frozen_model_results"]
}

checkpoint_hashes_match = all(
    Path(item["path"]).is_file()
    and sha256_file(Path(item["path"])) == item["sha256"]
    for item in freeze["checkpoints"]
)

report_files_exist = all(
    (
        output
        / f"stride_{int(item['stride']):02d}"
        / item["model_type"]
        / "comparative_test_report.json"
    ).is_file()
    for item in suite["all_frozen_model_results"]
)

prediction_files_exist = all(
    (
        output
        / f"stride_{int(item['stride']):02d}"
        / item["model_type"]
        / "comparative_test_predictions.csv"
    ).is_file()
    for item in suite["all_frozen_model_results"]
)

print("Freeze exists:", freeze_path.is_file())
print("Suite exists:", suite_path.is_file())
print("Completion lock exists:", lock_path.is_file())
print("Suite status:", suite["status"])
print("Independent confirmation:", suite["independent_confirmation"])
print("Frozen model results:", len(suite["all_frozen_model_results"]))
print("All 15 configurations evaluated:", observed == expected)
print("Baseline stride count:", len(suite["baseline_results"]))
print("All checkpoint hashes still match:", checkpoint_hashes_match)
print("All comparative reports exist:", report_files_exist)
print("All prediction files exist:", prediction_files_exist)
print("Freeze hash matches completion lock:",
      sha256_file(freeze_path) == lock["freeze_record_sha256"])
print("Suite hash matches completion lock:",
      sha256_file(suite_path) == lock["suite_report_sha256"])
print("Rerun permitted:", lock["rerun_permitted"])
print("No retraining after test:", suite["no_retraining_after_test"])
print("Completion status:", lock["status"])

Freeze exists: True
Suite exists: True
Completion lock exists: True
Suite status: post_test_comparative
Independent confirmation: False
Frozen model results: 15
All 15 configurations evaluated: True
Baseline stride count: 5
All checkpoint hashes still match: True
All comparative reports exist: True
All prediction files exist: True
Freeze hash matches completion lock: True
Suite hash matches completion lock: True
Rerun permitted: False
No retraining after test: True
Completion status: complete


In [11]:
import shutil
from pathlib import Path

results = Path(
    "/kaggle/working/k3_90to30_supervised_results_v1_1"
)

archive = shutil.make_archive(
    "/kaggle/working/k3_90to30_supervised_results_v1_1",
    "zip",
    results.parent,
    results.name
)

print(archive)

/kaggle/working/k3_90to30_supervised_results_v1_1.zip


In [12]:
from pathlib import Path
from zipfile import ZipFile
import hashlib

archive = Path(
    "/kaggle/working/k3_90to30_supervised_results_v1_1.zip"
)

digest = hashlib.sha256()

with archive.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(chunk)

with ZipFile(archive) as z:
    names = z.namelist()
    corrupt_file = z.testzip()

print("Archive exists:", archive.is_file())
print("Archive size bytes:", archive.stat().st_size)
print("Archive size MB:", round(archive.stat().st_size / (1024**2), 2))
print("Archive SHA-256:", digest.hexdigest())
print("Total archived files:", len(names))
print("Checkpoint files:", sum(n.endswith(".pt") for n in names))
print("JSON files:", sum(n.endswith(".json") for n in names))
print("CSV files:", sum(n.endswith(".csv") for n in names))
print(
    "Completion lock included:",
    any(n.endswith("TEST_EVALUATION_COMPLETE.json") for n in names)
)
print(
    "Freeze record included:",
    any(n.endswith("PRE_TEST_FREEZE.json") for n in names)
)
print("Corrupt file:", corrupt_file)

Archive exists: True
Archive size bytes: 212277909
Archive size MB: 202.44
Archive SHA-256: 5cd5b96a1f154cc766554c4099f80ddd11f5bf32ae629342e7287d40bb51bae6
Total archived files: 105
Checkpoint files: 15
JSON files: 39
CSV files: 30
Completion lock included: True
Freeze record included: True
Corrupt file: None


In [13]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

display(
    FileLink(
        "k3_90to30_supervised_results_v1_1.zip",
        result_html_prefix="Download frozen experiment results: "
    )
)

/kaggle/working/k3_90to30_supervised_results_v1_1.zip

In [ ]:
import shutil
results = Path('/kaggle/working/k3_90to30_supervised_results_v1_1')
archive = shutil.make_archive('/kaggle/working/k3_90to30_supervised_results_v1_1', 'zip', results.parent, results.name)
print(archive)